# TP1_4 : Profiler

**Changements par rapport à la version précédente :**
1. **Ajout de nvidia-smi et CEEMS**
2. **Ajout de Trace-ml-ai**
3. **Ajout de Perfetto** comme visualiseur principal, TensorBoard conservé mais déprimé 

**Portée mono-GPU.** Le Jour 1 tourne sur 1 GPU.

### Environnement de calcul

Les fonctions *python* de gestion de queue SLURM dévelopées par l'IDRIS et les fonctions dédiées à la formation DLO-JZ sont à importer.

Le module d'environnement pour les *jobs* et la taille des images sont fixés pour ce *notebook*.

**TODO :** choisir un *pseudonyme* (maximum 5 caractères) pour vous différencier dans la queue SLURM et dans les outils collaboratifs pendant la formation et la compétition.

In [4]:
from idr_pytools import display_slurm_queue, gpu_jobs_submitter, search_log
from dlojz_tools import controle_technique, compare, GPU_underthehood, plot_accuracy, lrfind_plot, imagenet_starter
MODULE = 'pytorch-gpu/py3/2.8.0'
image_size = 224
account = 'sos@a100'
name = 'test'   ## Pseudonyme à choisir

---
## Étage 0 — Avez-vous allumé votre GPU ?

In [2]:
n_gpu = 1
command = './dlojz1_3.py -b 512 --image-size 224'
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                           account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

batch job 0: 1 GPUs distributed on 1 nodes with 1 tasks / 1 gpus per node and 8 cpus per task
Submitted batch job 1178243
jobid = ['1178243']




Avant d'instrumenter quoi que ce soit, on vérifie la chose la plus simple : **le GPU travaille-t-il ?**



| Outil | Quand | Ce qu'il donne |
|---|---|---|
| `nvidia-smi` | **pendant** le job, sur le nœud de calcul | une vue instantanée ou continue du GPU |
| CEEMS | **pendant et après** le job, depuis un navigateur | l'historique complet du job : CPU, GPU, mémoire, énergie |

L'« utilisation GPU » affichée par ces outils mesure seulement la **fraction du temps pendant laquelle au moins un kernel s'exécute**. Elle ne dit pas si ce kernel fait du calcul utile. Un GPU peut afficher 100 % d'utilisation tout en exploitant une faible part de sa puissance de calcul.


### Se connecter au nœud de calcul de son job



`nvidia-smi` doit être lancé **sur le nœud où tourne votre job**. Ce n'est possible que pendant que le job est en cours d'exécution.

1. Depuis un terminal sur la frontale, repérer le nœud attribué à votre job (colonne `NODELIST`) :

```bash
squeue --me
```

2. Se connecter à ce nœud (remplacer `jzxh042` par votre nœud) :

```bash
ssh jzxh042
```

Le terminal est maintenant sur le nœud de calcul. Toutes les commandes `nvidia-smi` qui suivent se lancent depuis ce terminal.



```bash
nvidia-smi
```

À lire dans la sortie :

| Champ | Ce qu'il indique |
|---|---|
| `GPU-Util` | % du temps où un kernel tourne |
| `Memory-Usage` | mémoire GPU occupée / mémoire totale |
| `Pwr: Usage/Cap` | puissance consommée / puissance maximale autorisée |
| `Temp` | température du GPU |
| `Processes` | les processus qui utilisent le GPU  |


![screen_nvidia_smi](./images/nvidia-smi.png)

---
### `nvidia-smi --query-gpu` : enregistrer les mesures dans un fichier

Pour garder une trace et la tracer ensuite, on choisit les champs voulus et on écrit au format CSV, une ligne par seconde :

```bash
nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,clocks.sm,temperature.gpu \
           --format=csv -l 1 > gpu_log.csv
```

`Ctrl+C` arrête l'enregistrement. 

### CEEMS : accéder au tableau de bord

CEEMS (*Compute Energy & Emissions Monitoring Stack*) enregistre automatiquement les métriques de tous les jobs Slurm de **plus de 5 min**

**http://www.idris.fr/docs/jean-zay/performance_optimisation/ceems/**

1. Ouvrir **https://jean-zay-ceems.idris.fr**.
2. Se connecter avec **ses identifiants Jean Zay** (même authentification que JupyterHub).
3. Aller dans le menu **`Home > Dashboards > SLURM Job Metrics > User Job Summary`**.
4. Choisir dans le menu **`Account`** le type d'heures du projet (par exemple `…@a100` ou `…@h100`).

Pour obtenir en plus la trace d'exécution du job, ajouter dans le fichier de soumission Slurm :

```bash
export CEEMS_ENABLE_PROFILING=1
```



---
## Étage 1 — Quel goulot domine l'entraînement ? (TraceML)

L'étage 0 nous a dit si le GPU travaille. TraceML nous dit pourquoi il n'est pas plus efficace, en découpant le temps de chaque pas d'entraînement et en rendant un sorte de verdict ou plutôt une **hypothèse**.

| Composante | Ce qu'elle mesure |
|---|---|
| **Input Wait** | l'attente du batch suivant (DataLoader) |
| **H2D** | le transfert des données de la mémoire CPU vers le GPU |
| **Forward / Backward / Optimizer** | le calcul (regroupés sous **Compute**) |
| **Residual** | le temps du pas non attribué aux phases ci-dessus |

**Step Time = Input Wait + Traced Step Time** (le temps mesuré à l'intérieur du bloc `trace_step`).

Le rapport ajoute des métriques **système** (utilisation GPU, puissance, température), **processus** (RAM, mémoire CUDA) et **mémoire par pas** (pics d'allocation).

### Boucle d'entraînement avec TraceML


**1. Initialisation**, juste après les imports :
```python
import traceml_ai as traceml
traceml.init(mode="auto")
```
`mode="auto"` instrumente automatiquement le DataLoader, le forward, le backward, l'optimiseur et les transferts vers le GPU.

**2. Délimitation d'un pas d'entraînement** avec `trace_step` :
```python
for i, (images, labels) in enumerate(train_loader):
    ...
    with traceml.trace_step(model):
        images = images.to(gpu, memory_format=torch.channels_last)   # H2D
        labels = labels.to(gpu)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
```

Le chargement du batch (`enumerate(train_loader)`) reste hors du bloc : c'est lui que TraceML mesure comme **Input Wait**. Le transfert vers le GPU est placé dans le bloc pour être compté comme **H2D**.



In [2]:
n_gpu = 1
command = (f'traceml run dlojz1_4_traceml.py --html-report '
           '--run-name bs64_nw8_${SLURM_JOB_ID} '
           f'--args -b 64 --num-workers 8 --image-size {image_size} --test --test-nsteps 200')
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                           account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

batch job 0: 1 GPUs distributed on 1 nodes with 1 tasks / 1 gpus per node and 8 cpus per task
Submitted batch job 1318659
jobid = ['1318659']


### Analyse du rapport



1. **Verdict** : le goulot le plus probable
2. **Why** : les mesures qui justifient ce verdict.
3. **Next** : la première action ou investigation à tenter.
4. **Run context** : GPU, nombre de pas analysés, durée. 
5. **STEP TIMING** : la décomposition du temps d'un pas (Input Wait, H2D, Compute, Residual).
6. **Step Memory**, **System**, **Process** : mémoire GPU par pas, utilisation du nœud, ressources du processus.

| Verdict | Signification |
|---|---|
| **INPUT-BOUND** | l'attente des données représente une grande part du pas : le GPU est mal alimenté |
| **H2D-BOUND** | le transfert vers le GPU représente une grande part du pas |
| **COMPUTE-BOUND** | le calcul domine |
| **RESIDUAL-HEAVY** | une part notable du pas n'est attribuée à aucune phase |
| **BALANCED** | aucun goulot ne domine nettement |
| **INCOMPLETE DATA** | une phase n'a pas été mesurée  |


In [7]:
# Second run : batch size 512, 8 workers (même script, mêmes options TraceML).
n_gpu = 1
command = (f'traceml run dlojz1_4_traceml.py --html-report '
           '--run-name bs512_nw8_${SLURM_JOB_ID} '
           f'--args -b 512 --num-workers 8 --image-size {image_size} --test --test-nsteps 200')
jobid_512 = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                               account=account, time_max='00:10:00')
print(f'jobid_512 = {jobid_512}')

batch job 0: 1 GPUs distributed on 1 nodes with 1 tasks / 1 gpus per node and 8 cpus per task
Submitted batch job 1318706
jobid_512 = ['1318706']


**À retenir**

- **Comparer en images/s, pas en Step Time.** Un pas bs512 traite 8 fois plus d'images qu'un pas bs64 : son Step Time est forcément plus long, ce n'est pas une régression.
- Débit = batch size / Step Time (ici 409 → 671 img/s, ×1,64).
- **L'utilisation GPU ne mesure pas l'efficacité.** C'est la fraction du temps où un kernel tourne. Faible, elle ne prouve pas à elle seule un goulot sur les données, élevée elle ne prouve pas que le GPU calcule bien (ici elle baisse en bs512 alors que le débit augmente). La puissance consommée est plus parlante.
- **Input Wait** est le temps passé par la boucle à attendre le batch suivant. Ce n'est ni le temps de prétraitement des workers (ils travaillent en parallèle), ni directement le temps d'inactivité du GPU.
- **Residual** est le temps du pas non attribué à une phase
- **Corriger un goulot en révèle un autre :** en bs512, le GPU va plus vite par image, et c'est "le DataLoader qui devient la limite" (Input Wait 21 % → 36 %).
- **System = tout le nœud, Process = votre job.** Les lignes CPU et RAM de System incluent les autres jobs du nœud.

---
## Étage 2 — PyTorch Profiler 


On soumet **deux** jobs pour comparer deux régimes : **bs512** (GPU bien alimenté) et **bs64**
(GPU sous-alimenté). `--test-nsteps 8` car le profilage est actif sur 5 steps.

* Configurer le *profiler* et ses paramètres.

```python
    # pytorch profiler setup
	prof =  profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                    schedule=schedule(wait=1, warmup=1, active=5, repeat=1),
                    on_trace_ready=tensorboard_trace_handler('./profiler/' + os.environ['SLURM_JOB_NAME'] 
                                               + '_' + os.environ['SLURM_JOBID'] + '_bs' +
                                               str(mini_batch_size)  + '_is' + str(args.image_size)),
                    profile_memory=True,
                    record_shapes=False, 
                    with_stack=False,
                    with_flops=False
                    )
```

In [5]:
n_gpu = 1
command = [f'./dlojz1_4.py -b {bs} --image-size {image_size} --test --test-nsteps 8'
           for bs in (512, 64)]
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                           account=account, time_max='00:15:00')
print(f'jobid = {jobid}')

batch job 0: 1 GPUs distributed on 1 nodes with 1 tasks / 1 gpus per node and 8 cpus per task
Submitted batch job 1322287
Submitted batch job 1322288
jobid = ['1322287', '1322288']


In [5]:
jobid = ['46644', '46646']

In [5]:
!tree profiler/

profiler/
├── martin_37882_bs64_is224
│   └── jean-zay-iam01_1531034.1790030480086056526.pt.trace.json
├── martin_46609_bs64_is224
│   └── jean-zay-iam03_3817998.1790063716841452736.pt.trace.json
├── martin_46644_bs512_is224
│   └── jean-zay-iam03_3818631.1790063811679377294.pt.trace.json
├── martin_46646_bs64_is224
│   └── jean-zay-iam16_1168798.1790063830931539267.pt.trace.json
├── mem_snapshot_37882_bs64_is224.pickle
├── mem_snapshot_46609_bs64_is224.pickle
├── mem_snapshot_46644_bs512_is224.pickle
├── mem_snapshot_46646_bs64_is224.pickle
└── scalene_bs64.json

4 directories, 9 files


---
## Visualisation — Perfetto  

La trace `*.pt.trace.json` s'ouvre directement dans Perfetto, sans installation.

1. Récupérez `profiler/<name>_<jobid>_bs512_is224/*.pt.trace.json` sur votre poste
   
2. Ouvrir **https://ui.perfetto.dev** --> *Open trace file* --> sélectionner le `.json`.

**Se repérer**
- Navigation : `Z`/`S` zoom avant/arrière, `Q`/`D` déplacement, clic sur un bloc = détails en bas (nom, durée, arguments).
- Ligne **thread Python** (CPU) : les blocs `ProfilerStep#N` délimitent chaque pas
- Ligne **stream GPU** : les kernels réellement exécutés et les `Memcpy`.




---
## Visualisation — TensorBoard

L'intégration TensorBoard du PyTorch Profiler est **dépréciée**
et n'est plus activement maintenue. Certaines fonctions du *trace viewer* sont instables dans
les versions récentes. On la garde pour ses vues *Overview* / *Operator* / *GPU Kernel*
(pratiques pour les Tensor Cores).


**TODO** : visualiser cette trace grâce à l'application TensorBoard en suivant les étapes suivantes :
* ouvrir [jupyterhub.idris.fr](https://jupyterhub.idris.fr) dans un nouvel onglet du navigateur
* ouvrir une nouvelle instance JupyterHub en cliquant sur **Add New JupyterLab Instance**
* sélectionner **Spawn server on SLURM node** (on va réserver un GPU)
* sélectionner **Tensorboard** dans le menu **Frontend**
* définir le chemin des logs **$WORK/DLO-JZ/Jour1/tp_dlojz_jour1/profiler** dans **TensorBoard logs directory**

<div><img src="images/slurm_spawner_a100_tensorboard.png" width="550"></div>

* sélectionner l'option avancée `--partition=Octo-GPU A100 SXM4 with 80 GB GPU mem`

<div><img src="images/slurm_spawner_a100_tensorboard_advanced.png" width="550"></div>

* lancer l'instance TensorBoard
<div><img src="images/slurm_spawner_a100_tensorboard_start.png" width="550"></div>

**2. GPU Summary**

| Métrique | Définition | Interprétation |
|---|---|---|
| **GPU Utilization** | % du temps du pas où au moins un kernel tourne | même piège que nvidia-smi : ne dit rien de l'efficacité |
| **Est. SM Efficiency** | estimation de la part des SM réellement occupés, pondérée par la durée des kernels | un kernel avec moins de blocs que de SM laisse des SM vides : c'est ce qui distingue bs64 de bs512 |
| **Est. Achieved Occupancy** | estimation du nombre de warps actifs par SM par rapport au maximum | faible = pas assez de travail en parallèle pour masquer les latences |
| **Kernel Time using Tensor Cores** | % du temps des kernels passé dans des kernels Tensor Cores | doit être élevé avec l'AMP |

**3. Execution Summary** : temps moyen d'un pas et sa répartition (camembert).

| Catégorie | Ce qu'elle mesure |
|---|---|
| **Kernel** | calcul sur le GPU |
| **Memcpy / Memset** | copies et initialisations mémoire (dont le transfert hôte→GPU) |
| **Runtime** | appels CUDA côté CPU (lancement de kernels, synchronisations) |
| **DataLoader** | temps passé à récupérer le batch |
| **CPU Exec** | calcul CPU pendant lequel le GPU ne fait rien |
| **Other** | le reste|

**4. Step Time Breakdown** : la même répartition, pas par pas (barres empilées)